# Experiment 1: Two and three groups visualization

## Preliminaries

In [ ]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import badr  # noqa: E402
import itertools
import jax.numpy as jnp  # noqa: E402
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib import tri  # noqa: E402
import matplotlib.patheffects as pe  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

In [ ]:
# Helpers
def fairness_values(model, metric, dset, weights):
    model.set_group_weights(weights)
    model.fit(dset.X_train, dset.y_train, dset.groups)
    train_metric = float(metric.fun(model.coef_, dset, train_test="train").item())
    test_metric = float(metric.fun(model.coef_, dset, train_test="test").item())
    return {
        "train_metric": train_metric,
        "test_metric": test_metric,
    }


def erm_vals(model, metric, dset):
    weights = jnp.ones(len(dset.X_train_list)) / len(dset.X_train_list)
    dct = fairness_values(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def one_fit_vals(model, metric, dset):
    n = len(dset.X_train_list)
    vals = [
        (
            fairness_values(model, metric, dset, jnp.eye(n)[i])["train_metric"],
            jnp.eye(n)[i],
        )
        for i in range(n)
    ]
    _, best_w = min(vals, key=lambda x: x[0])
    dct = fairness_values(model, metric, dset, best_w)
    dct["weights"] = np.array(best_w, dtype=float)
    return dct


def balanced_vals(model, metric, dset):
    sizes = jnp.array([X.shape[0] for X in dset.X_train_list])
    weights = 1 / sizes
    weights = weights / weights.sum()
    dct = fairness_values(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def badr_vals(model, metric, dset, n_points=101):
    n = len(dset.X_train_list)
    grid = jnp.linspace(0.0, 1.0, num=n_points)
    best_val = jnp.inf
    best_w = None
    for w_tuple in itertools.product(grid, repeat=n):
        w = jnp.array(w_tuple)
        if jnp.isclose(w.sum(), 1.0):
            val = fairness_values(model, metric, dset, w)["train_metric"]
            if val < best_val:
                best_val = val
                best_w = w
    dct = fairness_values(model, metric, dset, best_w)
    dct["weights"] = np.array(best_w, dtype=float)
    return dct


def minmax_vals(model, metric, dset):
    if isinstance(model, badr.models.LogisticRegression):
        mm_model = badr.models.NonsmoothMinMaxLogisticRegression(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train_list, dset.y_train_list)
    elif isinstance(model, badr.models.SVM):
        mm_model = badr.models.NSMMSVM(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train, dset.y_train, dset.groups)
    elif isinstance(model, badr.models.RidgeRegression):
        mm_model = badr.models.NSMMRR(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train_list, dset.y_train_list)
    else:
        raise ValueError("Unsupported model type for minmax_vals")
    gw = jnp.asarray(mm_model.group_weights_, dtype=jnp.float64)
    dct = fairness_values(model, metric, dset, gw)
    dct["weights"] = np.array(gw, dtype=float)
    return dct


def lineplot(model, metric, dset):
    xs = jnp.linspace(0.0, 1.0, num=101)
    ys = []
    for x in xs:
        weights = jnp.array([x, 1 - x])
        dct = fairness_values(model, metric, dset, weights)
        ys.append(dct["train_metric"])
    ys = jnp.array(ys)
    return xs, ys

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.patheffects as pe


def _simplex_geometry():
    """Return reusable geometry helpers for barycentric↔2D conversions."""
    v1 = np.array([1, 0]) - (1 / 3) * np.ones(2)
    v2 = np.array([0, 1]) - (1 / 3) * np.ones(2)
    basis_change = np.array([v1, v2]).T
    inv_basis_change = np.linalg.inv(basis_change)

    c1 = np.array(
        [np.cos(np.pi / 2 - 2 * np.pi / 3), np.sin(np.pi / 2 - 2 * np.pi / 3)]
    )
    c2 = np.array([0.0, 1.0])
    c3 = np.array(
        [np.cos(np.pi / 2 + 2 * np.pi / 3), np.sin(np.pi / 2 + 2 * np.pi / 3)]
    )
    inv_2d_basis_change = np.linalg.inv(np.array([c1, c2]).T)

    scale_factor = np.max([np.linalg.norm(c1 - c2), np.linalg.norm(c1 - c3)])

    return {
        "basis_change": basis_change,
        "inv_basis_change": inv_basis_change,
        "inv_2d_basis_change": inv_2d_basis_change,
        "c1": c1,
        "c2": c2,
        "c3": c3,
        "scale_factor": scale_factor,
        "corners": np.array([c1, c2, c3]),
    }


def _from_2d_to_3d(u2, geom):
    coord_2d = geom["inv_2d_basis_change"] @ u2
    coord_3d = geom["basis_change"] @ coord_2d
    return np.array([coord_3d[0], coord_3d[1], -np.sum(coord_3d)]) + 1 / 3


def _from_3d_to_2d(u3, geom):
    restricted_u = u3[:-1] - (1 / 3) * np.ones(2)
    coord_2d = geom["inv_basis_change"] @ restricted_u
    return coord_2d[0] * geom["c1"] + coord_2d[1] * geom["c2"]


def plot_3d_simplex(
    simplex_entry,
    ax=None,
    fig=None,
    return_handles=False,
    method_colors=None,
    marker_styles=None,
    add_colorbar=False,
):
    """
    Draw a 3-group simplex heatmap using precomputed results.

    - Expects simplex_entry from build_simplex_results.
    - Does NOT recompute oracle values or mutate inputs.
    - Returns (tcf, handles, labels) if return_handles=True.
    """
    n_groups = simplex_entry.get("n_groups", 3)
    if n_groups != 3:
        raise ValueError(
            f"{n_groups} groups not supported for 3D simplex plot. Expected 3."
        )

    fair_name = simplex_entry.get("fair_name", "Fairness")
    grid_data = simplex_entry["grid"]
    x = grid_data["x"]
    y = grid_data["y"]
    Z_norm = grid_data["Z_norm"]
    triangles = grid_data["triangles"]
    corners = simplex_entry["geometry"]["corners"]

    markers = simplex_entry["markers"]

    # --- default colors / markers (match 1D plot)
    if method_colors is None:
        method_colors = {
            "Badr": "#C81919",
            "ERM": "#662C91",
            "Balanced": "#DA9F93",
            "One-Fit": "#A0AF63",
            "MinMax": "#6699CC",
        }

    if marker_styles is None:
        marker_styles = {
            "Badr": dict(marker="X", s=30, zorder=5, color=method_colors["Badr"]),
            "ERM": dict(marker="v", s=30, zorder=4, color=method_colors["ERM"]),
            "Balanced": dict(
                marker="^", s=30, zorder=4, color=method_colors["Balanced"]
            ),
            "One-Fit": dict(marker="o", s=30, zorder=4, color=method_colors["One-Fit"]),
            "MinMax": dict(marker="p", s=30, zorder=4, color=method_colors["MinMax"]),
        }

    # --- axis handling
    if ax is None:
        if fig is None:
            fig, ax = plt.subplots(figsize=(4, 4))
        else:
            ax = fig.add_subplot(111)

    # --- plotting: heatmap + contours
    triang_obj = tri.Triangulation(x, y, triangles=triangles)
    tcf = ax.tripcolor(triang_obj, Z_norm, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.tricontour(triang_obj, Z_norm, levels=15, colors="k", linewidths=0.3)

    # --- markers: match 1D legend labels & colors
    sc_handles = {}
    for key, label in [
        ("Badr", "badr"),
        ("ERM", "Uniform sampling"),
        ("Balanced", "Balanced sampling"),
        ("One-Fit", "One-group fitting"),
        ("MinMax", "Minimax fairness"),
    ]:
        coord = markers[key]
        sc_handles[key] = ax.scatter(
            coord[0], coord[1], label=label, **marker_styles[key]
        )

    # give all markers a dark outline (helps on top of colormap)
    for key, lw in [
        ("ERM", 1.5),
        ("Balanced", 1.5),
        ("One-Fit", 1.5),
        ("MinMax", 1.5),
        ("Badr", 1.5),
    ]:
        sc_handles[key].set_path_effects(
            [pe.withStroke(linewidth=lw, foreground="black")]
        )

    # triangle frame
    ax.add_patch(plt.Polygon(corners, fill=False, color="k"))
    ax.set_aspect("equal")

    pad = 0.105
    ax.set_xlim(corners[:, 0].min() - pad, corners[:, 0].max() + pad)
    ax.set_ylim(corners[:, 1].min() - pad, corners[:, 1].max() + pad)

    ax.set_title(f"{fair_name}")
    ax.grid(False)

    # no ticks
    ax.set_xticks([])
    ax.set_yticks([])

    # optional colorbar
    if add_colorbar and fig is not None:
        cbar = fig.colorbar(tcf, ax=ax)
        cbar.set_label(r"Metric (normalized)")

    legend_handles = [
        sc_handles["Badr"],
        sc_handles["ERM"],
        sc_handles["Balanced"],
        sc_handles["One-Fit"],
        sc_handles["MinMax"],
    ]
    legend_labels = [h.get_label() for h in legend_handles]

    if return_handles:
        return tcf, legend_handles, legend_labels
    return tcf

## Two groups

In [ ]:
datasets2 = [
    (
        "ACSEmployment (South Dakota, 2014)",
        badr.datasets.fetch_ACSEmployment(states="SD", year=2014),
    ),
    ("ACSEmployment", badr.datasets.fetch_ACSEmployment(states="WY", year=2016)),
    (
        "ACSTravelTime (Vermont, 2015)",
        badr.datasets.fetch_ACSTravelTime(states="VT", year=2015),
    ),
]
model = badr.models.LogisticRegression(l2_reg=1e-2)
DemParity = badr.metrics.DemographicParity()
DispZafar = badr.metrics.DisparateMistreatment()
EqualOpp = badr.metrics.EqualOpportunity()
DemParity.set_model(model)
DispZafar.set_model(model)
EqualOpp.set_model(model)
metrics2 = [
    ("Demographic Parity", DemParity),
    ("Disparate Mistreatment", DispZafar),
    ("Equal Opportunity", EqualOpp),
]
results2 = {}
for (dset_name, dset), (metric_name, metric) in itertools.product(datasets2, metrics2):
    key = (dset_name, metric_name)
    results2[key] = {
        "ERM": erm_vals(model, metric, dset),
        "One-Fit": one_fit_vals(model, metric, dset),
        "Balanced": balanced_vals(model, metric, dset),
        "Badr": badr_vals(model, metric, dset, n_points=101),
        "MinMax": minmax_vals(model, metric, dset),
        "LinePlot": lineplot(model, metric, dset),
    }

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp


def plot_results_grid(results, datasets, metrics, alpha=0.8):
    """
    results: dict keyed by (dset_name, metric_name)
    datasets: iterable of (dset_name, dset)
    metrics: iterable of (metric_name, metric)
    """

    from tueplots import figsizes, axes as tp_axes

    n_d = len(datasets)
    n_m = len(metrics)
    with plt.rc_context(figsizes.jmlr2001(nrows=n_d, ncols=n_m)):
        plt.rcParams.update(tp_axes.tick_direction(x="out", y="out"))
        plt.rcParams["font.family"] = "Open Sans"
        plt.rcParams["font.weight"] = "light"
        plt.rcParams["font.size"] = 9.95
        plt.rcParams["axes.facecolor"] = "white"
        plt.rcParams["figure.dpi"] = 200

        fig, axes_arr = plt.subplots(
            nrows=n_d,
            ncols=n_m,
            figsize=(8.5, 5.0),
        )

        # key_name in results, pretty legend label
        methods = [
            ("ERM", "Uniform sampling"),
            ("Balanced", "Balanced sampling"),
            ("One-Fit", "One-group fitting"),
            ("MinMax", "Minimax fairness"),
            ("Badr", "badr"),
        ]

        # fixed per-method colors (Badr stands out most)
        method_colors = {
            "ERM": "#662C91",
            "Balanced": "#DA9F93",
            "One-Fit": "#A0AF63",
            "MinMax": "#6699CC",
            "Badr": "#C81919",
        }

        # marker / size / zorder settings (now with explicit colors)
        marker_styles = {
            "ERM": dict(
                marker="v",
                s=30,
                zorder=3,
                color=method_colors["ERM"],
                alpha=alpha,
                lw=0.25,
                edgecolor="black",
            ),
            "Balanced": dict(
                marker="^",
                s=30,
                zorder=3,
                color=method_colors["Balanced"],
                alpha=alpha,
                lw=0.25,
                edgecolor="black",
            ),
            "One-Fit": dict(
                marker="o",
                s=30,
                zorder=4,
                color=method_colors["One-Fit"],
                alpha=alpha,
                lw=0.25,
                edgecolor="black",
            ),
            "MinMax": dict(
                marker="p",
                s=30,
                zorder=3,
                color=method_colors["MinMax"],
                alpha=alpha,
                lw=0.25,
                edgecolor="black",
            ),
            # Badr: slightly larger, on top, bright color, with black edge to pop
            "Badr": dict(
                marker="X",
                alpha=alpha,
                s=30,
                zorder=5,
                color=method_colors["Badr"],
                lw=0.25,
                edgecolor="black",
            ),
        }

        legend_handles = []
        legend_labels = []

        axes_arr = np.asarray(axes_arr).reshape(n_d, n_m)

        for i, (dset_name, _dset) in enumerate(datasets):
            for j, (metric_name, _metric) in enumerate(metrics):
                ax = axes_arr[i, j]

                key = (dset_name, metric_name)
                res = results2[key]

                xs, ys = res["LinePlot"]

                ys_min = float(jnp.min(ys))
                ys_max = float(jnp.max(ys))
                denom = ys_max - ys_min
                if abs(denom) < 1e-12:
                    ys_norm = jnp.zeros_like(ys)
                else:
                    ys_norm = (ys - ys_min) / denom

                # keep the line neutral so method colors dominate
                ax.plot(xs, ys_norm, linewidth=1.0, color="grey")

                for key_name, label in sorted(methods):
                    dct = res[key_name]
                    w = dct["weights"][0]
                    idx = int(jnp.argmin(jnp.abs(xs - w)))
                    style = marker_styles[key_name]
                    sc = ax.scatter(xs[idx], ys_norm[idx], label=label, **style)

                    if label not in legend_labels:
                        legend_labels.append(label)
                        legend_handles.append(sc)

                if i == 0:
                    ax.set_title(metric_name)

        # axis labels per column/row instead of shared labels
        for j in range(n_m):
            axes_arr[-1, j].set_xlabel("group 1 weight")
        for i in range(n_d):
            axes_arr[i, 0].set_ylabel("fairness")

        # dataset names on the right, one per row (top to bottom)
        row_labels = [
            "ACSEmployment \n SD 2017",
            "ACSEmployment \n WY 2016",
            "ACSTravelTime \n VT 2015",
        ]
        for i in range(min(n_d, len(row_labels))):
            ax_right = axes_arr[i, -1]  # last column of this row
            ax_right.text(
                1.1,
                0.5,  # just to the right of the last axis
                row_labels[i],
                transform=ax_right.transAxes,
                rotation=270,  # vertical text
                va="center",
                ha="center",  # extend text further to the right
            )

        if legend_handles:
            fig.legend(
                legend_handles,
                legend_labels,
                loc="lower center",
                ncol=len(legend_labels),
                bbox_to_anchor=(0.5, -0.07),
            )
        plt.savefig("../../figures/experiment_1_2gr.pdf", bbox_inches="tight")
        plt.show()


plot_results_grid(results2, datasets2, metrics2, alpha=0.9)

## Three groups

In [ ]:
import itertools

model = badr.models.SVM(l2_reg=1e0)

IndFair = badr.metrics.IndividualFairness()
IndFair.set_model(model)
EqOdds = badr.metrics.EqualizedOdds()
EqOdds.set_model(model)
EqualOpp = badr.metrics.EqualOpportunity()
EqualOpp.set_model(model)

metrics = [
    ("Individual Fairness", IndFair),
    ("Equalized Odds", EqOdds),
    ("Equal Opportunity", EqualOpp),
]

datasets = [
    (
        "ACSTravelTime (Vermont, 2016)",
        badr.datasets.fetch_ACSTravelTime(states="VT", year=2016, n_groups=3),
    ),
    (
        "ACSEmployment (North Dakota, 2018)",
        badr.datasets.fetch_ACSEmployment(states="ND", year=2018, n_groups=3),
    ),
    (
        "ACSEmployment (West Virginia, 2015)",
        badr.datasets.fetch_ACSEmployment(states="WV", year=2015, n_groups=3),
    ),
]


def _compute_simplex_entry(oracle, geom, metric_name, num_points):
    n_groups = getattr(oracle, "n_groups", 3)
    if n_groups != 3:
        raise ValueError(
            f"{n_groups} groups not supported for 3D simplex plot. Expected 3."
        )

    metric = getattr(oracle, "metric", None)
    fair_name = getattr(metric, "name", metric_name)

    xg = np.linspace(0.0, 1.0, num=num_points)
    yg = np.linspace(0.0, 1.0, num=num_points)
    b1, b2 = np.meshgrid(xg, yg)
    b3 = 1.0 - b1 - b2
    mask = (b1 + b2 <= 1.0) & (b1 >= 0.0) & (b2 >= 0.0)
    b1m, b2m, b3m = b1[mask], b2[mask], b3[mask]
    denom = b1m + b2m + b3m

    scale_factor = geom["scale_factor"]
    x = scale_factor * (0.5 * (2 * b2m + b3m) / denom - 0.5)
    y = scale_factor * ((np.sqrt(3) / 2) * b3m / denom - 1.0 / (2 * np.sqrt(3)))

    grid_weights = np.array(
        [_from_2d_to_3d(np.array([xi, yi]), geom) for xi, yi in zip(x, y)], dtype=float
    )
    Z = np.array([float(oracle.fun(w3)) for w3 in grid_weights], dtype=float)

    z_min = float(np.min(Z))
    z_max = float(np.max(Z))
    denom_norm = z_max - z_min
    if denom_norm < 1e-12:
        Z_norm = np.zeros_like(Z)
    else:
        Z_norm = (Z - z_min) / denom_norm

    min_idx = int(np.argmin(Z))
    badr_coord = np.array([x[min_idx], y[min_idx]], dtype=float)

    vertices = np.eye(3)
    vertex_coords_2d = np.array([_from_3d_to_2d(v, geom) for v in vertices])
    vertex_values = np.array([float(oracle.fun(v)) for v in vertices], dtype=float)
    one_idx = int(np.argmin(vertex_values))
    one_coord = vertex_coords_2d[one_idx]
    uniform_coord = _from_3d_to_2d(np.array([1 / 3, 1 / 3, 1 / 3], dtype=float), geom)

    minmax_weights = np.asarray(
        minmax_vals(oracle.model, oracle.metric, oracle.dset)["weights"], dtype=float
    )
    minmax_coord = _from_3d_to_2d(minmax_weights, geom)

    balanced_weights = np.asarray(
        balanced_vals(oracle.model, oracle.metric, oracle.dset)["weights"], dtype=float
    )
    balanced_coord = _from_3d_to_2d(balanced_weights, geom)

    used = {}

    def _nudge_if_duplicate(coord, delta=0.015):
        key = tuple(np.round(coord, 10))
        cnt = used.get(key, 0)
        used[key] = cnt + 1
        if cnt == 0:
            return coord
        return coord + cnt * delta * np.array([1.0, 1.0])

    badr_coord = _nudge_if_duplicate(badr_coord)
    one_coord = _nudge_if_duplicate(one_coord)
    uniform_coord = _nudge_if_duplicate(uniform_coord)
    minmax_coord = _nudge_if_duplicate(minmax_coord)
    balanced_coord = _nudge_if_duplicate(balanced_coord)

    triangulation = tri.Triangulation(x, y)

    return {
        "fair_name": fair_name,
        "metric_name": metric_name,
        "n_groups": n_groups,
        "grid": {
            "x": x,
            "y": y,
            "Z": Z,
            "Z_norm": Z_norm,
            "z_min": z_min,
            "z_max": z_max,
            "triangles": triangulation.triangles,
        },
        "markers": {
            "ERM": uniform_coord,
            "Balanced": balanced_coord,
            "One-Fit": one_coord,
            "MinMax": minmax_coord,
            "Badr": badr_coord,
        },
        "geometry": {"corners": geom["corners"]},
    }


def build_simplex_results(datasets, metrics, model, num_points=200):
    """
    Precompute simplex heatmap data for each (dataset, metric) pair.

    The returned structure contains only numpy arrays / metadata so that
    downstream plotting never re-runs the expensive oracle evaluations.
    """
    geom = _simplex_geometry()
    results = {}
    for (dset_name, dset), (metric_name, metric) in itertools.product(
        datasets, metrics
    ):
        metric.set_model(model)
        oracle = badr.oracles.ImplicitOracle(dset, model, metric)
        results[(dset_name, metric_name)] = _compute_simplex_entry(
            oracle, geom, metric_name, num_points
        )
    return results


results_3d = build_simplex_results(datasets, metrics, model, num_points=125)

In [ ]:
def plot_results_grid_simplex(results, datasets, metrics):
    """
    Render a grid of precomputed simplex plots.

    rows   = datasets
    columns= metrics
    """
    method_colors = {
        "Badr": "#C81919",
        "ERM": "#662C91",
        "Balanced": "#DA9F93",
        "One-Fit": "#A0AF63",
        "MinMax": "#6699CC",
    }
    alpha = 0.8
    marker_styles = {
        "Badr": dict(
            marker="x", s=30, zorder=6, alpha=alpha, color=method_colors["Badr"]
        ),
        "ERM": dict(
            marker="v", s=30, zorder=4, alpha=alpha, color=method_colors["ERM"]
        ),
        "Balanced": dict(
            marker="^", s=30, zorder=4, alpha=alpha, color=method_colors["Balanced"]
        ),
        "One-Fit": dict(
            marker=".", s=30, zorder=5, alpha=alpha, color=method_colors["One-Fit"]
        ),
        "MinMax": dict(
            marker="p", s=30, zorder=4, alpha=alpha, color=method_colors["MinMax"]
        ),
    }

    from tueplots import figsizes, axes as tp_axes

    n_d = len(datasets)
    n_m = len(metrics)
    with plt.rc_context(figsizes.jmlr2001(nrows=n_d, ncols=n_m)):
        plt.rcParams.update(tp_axes.tick_direction(x="out", y="out"))
        plt.rcParams["font.family"] = "Open Sans"
        plt.rcParams["font.weight"] = "light"
        plt.rcParams["font.size"] = 9.95
        plt.rcParams["axes.facecolor"] = "white"
        plt.rcParams["figure.dpi"] = 200

        fig, axes_arr = plt.subplots(
            nrows=n_d,
            ncols=n_m,
            figsize=(8.5, 5.0),
        )

        axes_arr = np.asarray(axes_arr).reshape(n_d, n_m)

        legend_handles = None
        legend_labels = None

        # store 1 representative mappable (for global colorbar)
        global_tcf = None

        for i, (dset_name, _dset) in enumerate(datasets):
            for j, (metric_name, _metric) in enumerate(metrics):
                ax = axes_arr[i, j]

                key = (dset_name, metric_name)
                entry = results[key]

                tcf, handles, labels = plot_3d_simplex(
                    entry,
                    ax=ax,
                    fig=fig,
                    return_handles=True,
                    method_colors=method_colors,
                    marker_styles=marker_styles,
                    add_colorbar=False,  # <-- disable per-axis colorbars
                )

                # store mappable for global colorbar later
                if global_tcf is None:
                    global_tcf = tcf

                if legend_handles is None:
                    legend_handles = handles
                    legend_labels = labels

                if i == 0:
                    ax.set_title(metric_name)
                else:
                    ax.set_title("")

        # --- GLOBAL COLORBAR HERE ---
        if global_tcf is not None:
            cbar = fig.colorbar(
                global_tcf,
                ax=axes_arr.ravel().tolist(),
                fraction=0.03,
                pad=0.02,
                location="right",
            )
            cbar.ax.tick_params(labelsize=7)
            # cbar.set_label("Metric (norm.)", fontsize=7)

        # row labels on right
        row_labels = [
            "ACSTravelTime \n VT 2016",
            "ACSEmployment \n ND 2018",
            "ACSEmployment \n WV 2015",
        ]
        for i in range(min(n_d, len(row_labels))):
            ax_right = axes_arr[i, -1]
            ax_right.text(
                1.15,
                0.5,
                row_labels[i],
                transform=ax_right.transAxes,
                rotation=270,
                va="center",
                ha="center",
            )

        if legend_handles is not None:
            fig.legend(
                legend_handles,
                legend_labels,
                loc="lower center",
                ncol=len(legend_labels),
                bbox_to_anchor=(0.5, -0.10),
            )

        plt.savefig("../../figures/experiment_1_3gr.pdf", bbox_inches="tight")
        plt.show()


plot_results_grid_simplex(results_3d, datasets, metrics)